# Pythia RBP Binary Classification

This notebook demonstrates training and evaluation of **PythiaRBP** for RNA-binding protein binding prediction using toy stem-loop sequence data.

In [ ]:
import sys
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from torch.utils.data import DataLoader, TensorDataset

# Add parent to path so pythia is importable without install
sys.path.insert(0, str(Path('..').resolve().parent))

from pythia.configs import RBPModelConfig, RBPTrainConfig
from pythia.models.rbp import PythiaRBP

matplotlib.use('Agg')
sns.set_theme(style='whitegrid', context='notebook')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('Imports OK')

## 1. Generate Toy Stem-Loop Data

**Positive class**: sequences with a 4-bp stem and 6-nt random loop.
- Stem: pos 0-3 pair with pos 13-10 (Watson-Crick reverse complement)
- Total length: 14 nt
- Pattern: `NNNN + NNNNNN + RC(NNNN)` where stem nucleotides form valid WC pairs

**Negative class**: fully random 14-nt sequences.

In [ ]:
NUCLEOTIDES = ['A', 'C', 'G', 'U']
RNA_MAP = {'A': 0, 'C': 1, 'G': 2, 'U': 3}
RC_MAP = {'A': 'U', 'U': 'A', 'C': 'G', 'G': 'C'}
SEQ_LEN = 14
STEM_LEN = 4
LOOP_LEN = 6


def random_rna(length: int) -> str:
    return ''.join(random.choice(NUCLEOTIDES) for _ in range(length))


def make_stem_loop() -> str:
    """Generate a positive stem-loop sequence of length 14."""
    stem = [random.choice(NUCLEOTIDES) for _ in range(STEM_LEN)]
    loop = [random.choice(NUCLEOTIDES) for _ in range(LOOP_LEN)]
    rc_stem = [RC_MAP[n] for n in reversed(stem)]
    return ''.join(stem + loop + rc_stem)


def one_hot(seq: str, length: int = SEQ_LEN) -> np.ndarray:
    arr = np.zeros((4, length), dtype=np.float32)
    for i, ch in enumerate(seq[:length]):
        idx = RNA_MAP.get(ch)
        if idx is not None:
            arr[idx, i] = 1.0
    return arr


N_POS = 2000
N_NEG = 2000

pos_seqs = [make_stem_loop() for _ in range(N_POS)]
neg_seqs = [random_rna(SEQ_LEN) for _ in range(N_NEG)]

all_seqs = pos_seqs + neg_seqs
all_labels = [1] * N_POS + [0] * N_NEG

idx_perm = np.random.permutation(len(all_seqs))
all_seqs = [all_seqs[i] for i in idx_perm]
all_labels = [all_labels[i] for i in idx_perm]

# One-hot encode
X = np.stack([one_hot(s) for s in all_seqs])  # (N, 4, 14)
y = np.array(all_labels, dtype=np.int64)

# Train/val/test splits
n = len(X)
n_train = int(0.7 * n)
n_val = int(0.15 * n)

X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train:n_train + n_val], y[n_train:n_train + n_val]
X_test, y_test = X[n_train + n_val:], y[n_train + n_val:]

print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')
print(f'Example positive: {pos_seqs[0]}')
print(f'Example negative: {neg_seqs[0]}')

## 2. Build DataLoaders

In [ ]:
def make_loader(X, y, batch_size=64, shuffle=True):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


train_loader = make_loader(X_train, y_train, batch_size=64, shuffle=True)
val_loader = make_loader(X_val, y_val, batch_size=128, shuffle=False)
test_loader = make_loader(X_test, y_test, batch_size=128, shuffle=False)
print('Loaders ready')

## 3. Build PythiaRBP Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

cfg = RBPModelConfig(
    inputsize=SEQ_LEN,
    init_channels=32,
    dp=0.25,
    dil_start=2,
    dil_end=6,
    bulge_size=2,
)

model = PythiaRBP(cfg).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f'Parameters: {trainable:,} trainable, {frozen:,} frozen')

## 4. Train with Early Stopping

In [ ]:
import math

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
EPOCHS = 30
PATIENCE = 5

train_losses, val_losses, val_accs = [], [], []
best_val_loss = float('inf')
best_state = None
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    ep_loss = 0.0
    for x, lab in train_loader:
        x, lab = x.to(device), lab.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, lab)
        loss.backward()
        optimizer.step()
        ep_loss += loss.item()
    train_losses.append(ep_loss / len(train_loader))

    model.eval()
    vl, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, lab in val_loader:
            x, lab = x.to(device), lab.to(device)
            logits = model(x)
            vl += criterion(logits, lab).item()
            correct += (logits.argmax(1) == lab).sum().item()
            total += len(lab)
    val_loss = vl / len(val_loader)
    val_acc = correct / total
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f'Epoch {epoch:02d} | train_loss={train_losses[-1]:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

model.load_state_dict(best_state)
print('\nTraining complete.')

## 5. Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs_ran = range(1, len(train_losses) + 1)

axes[0].plot(epochs_ran, train_losses, label='Train', lw=2)
axes[0].plot(epochs_ran, val_losses, label='Val', lw=2)
axes[0].set_xlabel('Epoch', fontweight='bold')
axes[0].set_ylabel('Cross-Entropy Loss', fontweight='bold')
axes[0].set_title('Training & Validation Loss', fontweight='bold')
axes[0].legend()

axes[1].plot(epochs_ran, val_accs, color='seagreen', lw=2)
axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_ylabel('Accuracy', fontweight='bold')
axes[1].set_title('Validation Accuracy', fontweight='bold')

for ax in axes:
    sns.despine(ax=ax)

fig.tight_layout()
plt.savefig('rbp_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: rbp_training_curves.png')

## 6. Evaluate on Test Set

In [ ]:
model.eval()
all_probs, all_true = [], []

with torch.no_grad():
    for x, lab in test_loader:
        x = x.to(device)
        logits = model(x)
        probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_probs.extend(probs.tolist())
        all_true.extend(lab.numpy().tolist())

y_true_np = np.array(all_true)
y_score_np = np.array(all_probs)

auroc = roc_auc_score(y_true_np, y_score_np)
aupr = average_precision_score(y_true_np, y_score_np)
acc = ((y_score_np > 0.5).astype(int) == y_true_np).mean()

print(f'Test AUROC: {auroc:.4f}')
print(f'Test AUPR:  {aupr:.4f}')
print(f'Test Acc:   {acc:.4f}')

## 7. ROC and PR Curves

In [ ]:
fpr, tpr, _ = roc_curve(y_true_np, y_score_np)
prec, rec, _ = precision_recall_curve(y_true_np, y_score_np)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, lw=2, color='steelblue', label=f'PythiaRBP (AUROC={auroc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate', fontweight='bold')
axes[0].set_ylabel('True Positive Rate', fontweight='bold')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].legend()

axes[1].plot(rec, prec, lw=2, color='tomato', label=f'PythiaRBP (AUPR={aupr:.3f})')
axes[1].axhline(y_true_np.mean(), color='k', ls='--', lw=1, label='Baseline')
axes[1].set_xlabel('Recall', fontweight='bold')
axes[1].set_ylabel('Precision', fontweight='bold')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend()

for ax in axes:
    sns.despine(ax=ax)

fig.tight_layout()
plt.savefig('rbp_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: rbp_performance.png')

## 8. Score Distribution by Class

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for cls, color, label in [(0, 'steelblue', 'Unbound'), (1, 'tomato', 'Bound')]:
    mask = y_true_np == cls
    ax.hist(y_score_np[mask], bins=40, alpha=0.6, color=color, label=label, density=True)

ax.set_xlabel('Predicted Binding Probability', fontweight='bold')
ax.set_ylabel('Density', fontweight='bold')
ax.set_title('Score Distribution by Class', fontweight='bold')
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('rbp_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: rbp_score_distribution.png')